# Phase 2.2：Dense 直觉、Cosine 和 RRF 融合

## 目标

不把 Dense 当作黑盒：先用二维向量理解 cosine，再用两个人工排名手写 RRF，最后接入生产融合函数。真实 BGE-M3 依赖是可选项，本课不会把模拟向量冒充模型成绩。

**本课交付：** `data/processed/phase2_rrf_demo.json`。

## Evidence Quest 任务卡：Phase 2.2：检索双雄对决

**你的身份：** 排名策略师  
**案件背景：** 关键词检索擅长精确命中，向量检索擅长语义相近。你要让两支队伍先各自展示排名，再观察 RRF 如何做裁判。

### 本关专业 Goal

用二维直觉实验理解 cosine 和 RRF，并诚实区分模拟结果与真实模型结果。

### 你要交付的作品

**BM25 vs Dense vs Hybrid 对决板**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：融合策略师  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. Dense 检索到底改变了什么？

Sparse/BM25 主要观察词是否出现；Dense 把文本映射到连续向量，向量方向可以表达训练数据学到的语义关系。我们先不下载模型，只用二维向量理解“相似度排序”这个机制。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase2.2'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase2.2
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 导入 NumPy，用数组表示二维向量。
import numpy as np

# 定义一个查询向量，假设它代表“猫”这一语义方向。
query_vector = np.array([1.0, 0.0])

# 定义三个文档向量，方向分别接近猫、猫的近义表达和汽车型号。
document_vectors = {"cat": np.array([0.9, 0.1]), "kitten": np.array([0.8, 0.3]), "car_model": np.array([0.1, 0.99])}

# 输出向量形状，确认每个向量都有两个维度。
print("query shape:", query_vector.shape)
print("document shape:", document_vectors["cat"].shape)

# 向量的维度是模型表示语义的坐标数量，真实模型通常远大于 2。
assert query_vector.shape == (2,)

query shape: (2,)
document shape: (2,)


In [4]:
# 定义 cosine 函数，比较两个向量的方向相似度。
def cosine_similarity(left, right):
    # 计算两个向量的点积。
    dot_product = np.dot(left, right)

    # 计算左向量的长度。
    left_norm = np.linalg.norm(left)

    # 计算右向量的长度。
    right_norm = np.linalg.norm(right)

    # 用点积除以长度乘积，得到方向相似度。
    return float(dot_product / (left_norm * right_norm))

# 计算查询与每篇文档的 cosine 分数。
dense_scores = {}
for document_id, vector in document_vectors.items():
    # 保存当前文档和查询向量的相似度。
    dense_scores[document_id] = cosine_similarity(query_vector, vector)

# 从高到低排序，得到 Dense 的人工排名。
dense_ranking = sorted(dense_scores, key=dense_scores.get, reverse=True)

# 输出分数和排名，观察语义方向如何影响顺序。
print(dense_scores)
print(dense_ranking)

# “cat” 应该比 “car_model” 更接近查询方向。
assert dense_ranking[0] == "cat"

{'cat': 0.9938837346736189, 'kitten': 0.9363291775690444, 'car_model': 0.1004987059618685}
['cat', 'kitten', 'car_model']


### 重要边界：模拟不是 BGE-M3

上面的向量是我们手工写的，用来理解数学。它没有经过文本模型训练，所以不能得出“Dense 召回率是多少”的结论。真实 BGE-M3 实验必须记录模型 ID、revision、设备、向量维度、归一化方法和数据集版本。

In [5]:
# 导入 importlib.util，用它检查可选包是否安装。
import importlib.util

# 检查 FlagEmbedding 是否存在，不在 Notebook 中偷偷下载模型。
flag_embedding_available = importlib.util.find_spec("FlagEmbedding") is not None

# 打印检查结果，让环境能力边界可见。
print("FlagEmbedding installed:", flag_embedding_available)

# 当前教学课只要求知道真实模型的入口，不要求网络下载才能完成。
print("真实 BGE-M3 入口：BGEM3FlagModel('BAAI/bge-m3')；需单独记录下载和运行条件。")

FlagEmbedding installed: False
真实 BGE-M3 入口：BGEM3FlagModel('BAAI/bge-m3')；需单独记录下载和运行条件。


## 2. 为什么不能直接把 BM25 分数和 cosine 相加？

BM25 分数可能是 0～若干，cosine 通常在 -1～1；两个分数的量纲和分布不同。直接相加等于默认它们已经校准，这个默认通常没有证据。

RRF 只使用排名：某文档在一条排名中第 1 名，就贡献 `1/(rrf_k+1)`；出现在另一条排名中，还会贡献第二份分数。

In [6]:
# 定义两路人工排名，模拟精确匹配和语义匹配互补。
bm25_ranking = ["exact_model", "semantic_doc", "unrelated"]
dense_ranking = ["semantic_doc", "exact_model", "another_doc"]

# 设置 RRF 的平滑常数，常见默认值是 60。
rrf_k = 60

# 创建空字典，用于累加每个文档的 RRF 分数。
rrf_scores = {}

# 把两路排名放入列表，统一处理。
ranking_lists = [bm25_ranking, dense_ranking]

# 逐路处理排名。
for ranking in ranking_lists:
    # enumerate 从 0 开始，所以 rank 加 1 才符合人类排名。
    for zero_based_rank, document_id in enumerate(ranking):
        # 把机器索引转换为第 1 名、第 2 名等。
        rank = zero_based_rank + 1

        # 计算当前文档在当前排名中的贡献。
        contribution = 1 / (rrf_k + rank)

        # 把贡献累加到文档总分。
        rrf_scores[document_id] = rrf_scores.get(document_id, 0.0) + contribution

# 按 RRF 分数从高到低得到融合排名。
manual_fused = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

# 打印每个文档的融合分数和最终顺序。
print(rrf_scores)
print(manual_fused)

# 两路都出现的文档应该获得两份贡献。
assert "exact_model" in manual_fused
assert "semantic_doc" in manual_fused

{'exact_model': 0.03252247488101534, 'semantic_doc': 0.03252247488101534, 'unrelated': 0.015873015873015872, 'another_doc': 0.015873015873015872}
['exact_model', 'semantic_doc', 'unrelated', 'another_doc']


## 3. 接入正式 RRF 函数

正式函数还会去重同一路排名中的重复 ID、保存最佳名次、稳定处理同分排序。现在把同样的输入交给生产模块，验证我们的理解没有偏离。

In [7]:
# 导入项目中的生产 RRF 函数。
from phase2_semantic_search.fusion import reciprocal_rank_fusion

# 传入带名称的两路排名，名称方便之后记录实验来源。
production_fused = reciprocal_rank_fusion({"bm25": bm25_ranking, "dense": dense_ranking}, rrf_k=rrf_k)

# 提取正式结果中的文档 ID 顺序。
production_ids = [item.doc_id for item in production_fused]

# 打印正式结果，观察它还提供了 best_rank 等解释字段。
for item in production_fused:
    # 输出文档 ID、融合分数和最佳名次。
    print(item.doc_id, round(item.score, 6), item.best_rank)

# 正式函数至少应该返回三篇不同文档。
assert len(production_ids) == 4

exact_model 0.032522 1
semantic_doc 0.032522 1
another_doc 0.015873 3
unrelated 0.015873 3


In [8]:
# 组合 RRF 教学记录，明确标记排名来自人工模拟。
rrf_record = {
    "experiment_type": "mechanism_demo",
    "ranking_sources": ["manual_bm25_like", "manual_dense_like"],
    "rrf_k": rrf_k,
    "rankings": {"bm25": bm25_ranking, "dense": dense_ranking},
    "fused_ids": production_ids,
    "note": "人工向量和人工排名用于理解机制，不代表 BGE-M3 质量指标。",
}

# 指定 RRF 教学记录路径。
rrf_path = ROOT / "data" / "processed" / "phase2_rrf_demo.json"

# 保存记录，防止把模拟实验误写成无条件结论。
rrf_path.write_text(json.dumps(rrf_record, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印交付路径。
print("已生成:", rrf_path)

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\phase2_rrf_demo.json


## 本课验收

- [ ] 能解释向量、点积、范数和 cosine 的关系。
- [ ] 能明确区分 Dense 原理模拟与真实 BGE-M3 实验。
- [ ] 能手写 RRF，并说明它为什么不需要比较原始分数量纲。
- [ ] 能读懂正式 RRF 返回的 `doc_id/score/best_rank`。
- [ ] 已生成 `phase2_rrf_demo.json`。

## Boss Challenge：手算两份排名的 RRF 分数，观察共同出现的文档为什么得到更高融合排名。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [9]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [10]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase2_rrf_demo.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\phase2_rrf_demo.json']
